# Lab 07 solution, notebook 2: let the specialists hand back, and carry a conversation through the manager

The worked answers to the two challenge tasks of `07b_agents_sdk_at_the_desk.ipynb`. The setup repeats the
notebook's lookups and agents, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from agents import set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

In [3]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [4]:
from agents import Agent, ModelSettings, Runner, function_tool
from openai.types.shared import Reasoning

SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))

invoices_agent = Agent(
    name="Invoices",
    handoff_description="Answers questions about a customer's invoices: how many, and their total.",
    instructions=("You answer questions about a customer's invoices. Look up every fact with your tools; "
                  "never answer from memory. Answer in one or two sentences."),
    model=MODEL,
    tools=[function_tool(find_customer), function_tool(get_invoices)],
    model_settings=SETTINGS,
)

reps_agent = Agent(
    name="Support reps",
    handoff_description="Answers who a customer's support rep is.",
    instructions=("You answer who looks after a customer. Look up every fact with your tools; "
                  "never answer from memory. Answer in one or two sentences."),
    model=MODEL,
    tools=[function_tool(find_customer), function_tool(get_support_rep)],
    model_settings=SETTINGS,
)

manager = Agent(
    name="Manager",
    instructions=("You help the support reps of a music shop. Ask the specialists for every fact; never answer from "
                  "memory. Answer in one or two sentences."),
    model=MODEL,
    tools=[
        invoices_agent.as_tool(
            tool_name="ask_invoices",
            tool_description="Ask the invoices specialist about a customer's invoices: how many, and their total.",
        ),
        reps_agent.as_tool(
            tool_name="ask_support_reps",
            tool_description="Ask the support-rep specialist who looks after a customer.",
        ),
    ],
    model_settings=SETTINGS,
)

triage = Agent(
    name="Triage",
    instructions=("You route each request to the right specialist. Hand off to Invoices for invoice counts and "
                  "totals, and to Support reps for who looks after a customer."),
    model=MODEL,
    handoffs=[invoices_agent, reps_agent],
    model_settings=SETTINGS,
)

PUJA = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"
print("agents ready:", invoices_agent.name, "|", reps_agent.name, "|", manager.name, "|", triage.name)

agents ready: Invoices | Support reps | Manager | Triage


## Task 1: Let the specialists hand back

In the notebook the handoff lost the rep because control went to `Invoices` and stayed there. Here each specialist
can hand back to a triage agent, and the triage can hand on to the other specialist.

`clone(...)` copies an agent and changes the fields we name. The three copies refer to each other, so we create the
two specialists first, then the triage with handoffs to both, then set each specialist's `handoffs` to the new
triage.

In [5]:
BACK = (" If the request also asks for something your tools cannot look up, hand off to Triage with what is "
        "still missing.")

invoices_back = invoices_agent.clone(instructions=invoices_agent.instructions + BACK)
reps_back = reps_agent.clone(instructions=reps_agent.instructions + BACK)
triage_back = triage.clone(
    instructions=triage.instructions + " When a specialist hands back, hand off to the specialist for what is "
                                       "still missing.",
    handoffs=[invoices_back, reps_back],
)
invoices_back.handoffs = [triage_back]
reps_back.handoffs = [triage_back]

for agent in [triage_back, invoices_back, reps_back]:
    print(agent.name, "->", [target.name for target in agent.handoffs])

Triage -> ['Invoices', 'Support reps']
Invoices -> ['Triage']
Support reps -> ['Triage']


Every agent can now reach every other through the triage. `max_turns=12` leaves room for several handoffs, since
each one is a turn of the agent that hands off.

In [6]:
result = await Runner.run(triage_back, PUJA, max_turns=12)

print(result.final_output)
print("names the rep:", "Jane Peacock" in result.final_output, "| last agent:", result.last_agent.name,
      "| requests:", result.context_wrapper.usage.requests)
print("handoffs:", " | ".join(f"{item.source_agent.name} -> {item.target_agent.name}"
                              for item in result.new_items if item.type == "handoff_output_item"))

Puja Srivastava has 6 invoices totaling 36.64, and her support rep is Jane Peacock.
names the rep: True | last agent: Support reps | requests: 8
handoffs: Triage -> Invoices | Invoices -> Triage | Triage -> Support reps


In this run the path was `Triage -> Invoices -> Triage -> Support reps`, the reply had all three facts, and it took
8 requests.

Handing back works, but not every time. In 20 runs of this hand-back design on the same model, 15 had all three
facts. In the others the reply had the invoices and no rep. A handoff gives the agent the whole conversation, and
the agent decides for itself what to do with it.

Compare with the specialists as tools in the notebook: the same 8 requests, with the manager in charge of the
answer the whole time. When a question needs several specialists and one answer, calling them as tools is the
simpler design.

## Task 2: Carry a conversation through the manager

The session belongs to the manager's run. The specialists run as tools and see only the `input` the manager writes
for them.

In [7]:
from agents import SQLiteSession

SESSION_FILE = ROOT / "07_OpenAI_Agents_SDK" / "outputs" / "sessions.db"
SESSION_FILE.parent.mkdir(exist_ok=True)
session = SQLiteSession("manager-puja-helena", SESSION_FILE)
await session.clear_session()

first = await Runner.run(manager, PUJA, session=session)
print("run 1:", first.final_output)
second = await Runner.run(manager, "And what about Helena Holý?", session=session)
print("run 2:", second.final_output)
print()
for item in second.new_items:
    if item.type == "tool_call_item":
        print("manager asked:", item.raw_item.name, item.raw_item.arguments)

run 1: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.


run 2: Helena Holý has 7 invoices totaling $49.62, and her support rep is Steve Johnson.

manager asked: ask_invoices {"input":"How many invoices does customer Helena Holý have, and what is their total amount?"}
manager asked: ask_support_reps {"input":"Who is the support rep for customer Helena Holý?"}


Run 2 has Helena's three facts: 7 invoices, \$49.62 and Steve Johnson. The manager read *And what about Helena
Holý?* together with the session, which held Puja's question and its answer, and wrote each specialist a complete
question: `ask_invoices` got *How many invoices does customer Helena Holý have, and what is their total amount?*

**Key lesson:** the session gave the manager the context, and the manager turned it into complete questions. A
specialist called as a tool never reads the conversation, so the manager's `input` has to carry everything the
specialist needs.